# IMPPORT ESSENTIAL LIBRARIES

In [ ]:
import torch
import torchaudio
import librosa

import pickle
from tqdm import tqdm

import os, glob
import pandas as pd
import numpy as np

from sklearn.preprocessing import LabelEncoder

import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.utils import Sequence
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import normalize

from tensorflow.keras.optimizers import Adam

2025-09-25 13:12:45.361025: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1758805965.702794      36 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1758805965.808342      36 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


# DATA OPTIMIZATION

## inforamtion about data

In [2]:
DATA_DIR = "/kaggle/input/speaker-recognition-cmu-arctic/train"
rows = []

for sp in sorted(os.listdir(DATA_DIR)):
    spath = os.path.join(DATA_DIR, sp)
    if not os.path.isdir(spath): continue
    wavs = glob.glob(os.path.join(spath, "*.wav"))
    for w in wavs:
        try:
            y, sr = librosa.load(w, sr=None)
            dur = librosa.get_duration(y=y, sr=sr)
        except:
            dur = None
        rows.append((sp, w, dur))
        
df = pd.DataFrame(rows, columns=["speaker","path","duration"])
print("Speakers:", df["speaker"].nunique())
print("Files per speaker:\n", df["speaker"].value_counts())
print("Durations stats:\n", df["duration"].describe())


Speakers: 18
Files per speaker:
 speaker
awb    910
jmk    909
aew    906
clb    906
rms    906
ksp    906
slt    906
bdl    906
lnh    905
rxr    533
eey    475
fem    474
ahw    474
axb    474
ljm    474
aup    474
slp    474
gka    474
Name: count, dtype: int64
Durations stats:
 count    12486.000000
mean         3.206057
std          0.918252
min          0.925000
25%          2.545062
50%          3.155063
75%          3.785000
max          7.665063
Name: duration, dtype: float64


## make all records be 3 sec

In [3]:
INPUT_DIR = "/kaggle/input/speaker-recognition-cmu-arctic/train"
OUTPUT_DIR = "preprocessed_train"

# same sample rate and duration
SAMPLE_RATE = 16000
TARGET_LEN = SAMPLE_RATE * 3

processed = 0
skipped = 0

for speaker in os.listdir(INPUT_DIR):
    speaker_path = os.path.join(INPUT_DIR, speaker)
    if not os.path.isdir(speaker_path):
        continue

#folder for each speaker
    speaker_out = os.path.join(OUTPUT_DIR, speaker)
    os.makedirs(speaker_out, exist_ok=True)

    for f in os.listdir(speaker_path):
        file_path = os.path.join(speaker_path, f)

        try:
            signal, sr = torchaudio.load(file_path)

# resample if needed
            if sr != SAMPLE_RATE:
                signal = torchaudio.transforms.Resample(orig_freq=sr, new_freq=SAMPLE_RATE)(signal)

            duration = signal.size(1) / SAMPLE_RATE

            if duration < 1.0:
                print(f" Skipped (too short): {f} ({duration:.2f}s)")
                skipped += 1
                continue

# padding
            if signal.size(1) < TARGET_LEN:
                pad_len = TARGET_LEN - signal.size(1)
                signal = torch.nn.functional.pad(signal, (0, pad_len))
                print(f" Padded: {f} ({duration:.2f}s → 3s)")

# crop if more than 3s
            else:
                signal = signal[:, :TARGET_LEN]
                print(f" Cropped: {f} ({duration:.2f}s → 3s)")

# save after preprocessing
            out_path = os.path.join(speaker_out, f)
            torchaudio.save(out_path, signal, SAMPLE_RATE)
            processed += 1

        except Exception as e:
            print(f" Error in {file_path}: {e}")

print(f"\n Processing finished! {processed} files saved in '{OUTPUT_DIR}', Skipped: {skipped}")


 Cropped: arctic_a0095.wav (4.26s → 3s)
 Padded: arctic_a0306.wav (2.98s → 3s)
 Cropped: arctic_a0051.wav (4.12s → 3s)
 Cropped: arctic_a0518.wav (3.00s → 3s)
 Padded: arctic_a0322.wav (2.81s → 3s)
 Padded: arctic_a0330.wav (2.92s → 3s)
 Cropped: arctic_a0349.wav (3.72s → 3s)
 Padded: arctic_a0173.wav (2.71s → 3s)
 Cropped: arctic_a0501.wav (4.01s → 3s)
 Padded: arctic_a0418.wav (2.74s → 3s)
 Padded: arctic_a0301.wav (2.72s → 3s)
 Cropped: arctic_a0233.wav (3.29s → 3s)
 Cropped: arctic_a0277.wav (4.51s → 3s)
 Cropped: arctic_a0168.wav (3.67s → 3s)
 Cropped: arctic_a0424.wav (4.20s → 3s)
 Cropped: arctic_a0472.wav (5.03s → 3s)
 Padded: arctic_a0122.wav (2.85s → 3s)
 Padded: arctic_a0409.wav (2.56s → 3s)
 Cropped: arctic_a0188.wav (3.27s → 3s)
 Cropped: arctic_a0006.wav (3.25s → 3s)
 Cropped: arctic_a0250.wav (5.53s → 3s)
 Padded: arctic_a0356.wav (2.78s → 3s)
 Cropped: arctic_a0451.wav (3.72s → 3s)
 Cropped: arctic_a0500.wav (3.90s → 3s)
 Cropped: arctic_a0575.wav (3.70s → 3s)
 Padded: 

In [4]:
DATA_DIR = "/kaggle/input/speaker-recognition-cmu-arctic/test"
OUTPUT_DIR = "preprocessed_test"

SAMPLE_RATE = 16000
TARGET_LEN = SAMPLE_RATE * 3
os.makedirs(OUTPUT_DIR, exist_ok=True)

processed = 0
skipped = 0

for root, dirs, files in os.walk(DATA_DIR):
    for f in files:
        if not f.endswith(".wav"):
            continue

        path = os.path.join(root, f)
        signal, sr = torchaudio.load(path)

        if sr != SAMPLE_RATE:
            resampler = torchaudio.transforms.Resample(sr, SAMPLE_RATE)
            signal = resampler(signal)

        duration = signal.size(1) / SAMPLE_RATE

        if duration < 1.0:
            print(f" Skipped (too short): {f} ({duration:.2f}s)")
            skipped += 1
            continue

        if signal.size(1) < TARGET_LEN:
            # padding
            pad_len = TARGET_LEN - signal.size(1)
            signal = torch.nn.functional.pad(signal, (0, pad_len))
            print(f" Padded: {f} ({duration:.2f}s → 3s)")
            
        else:
            # crop
            signal = signal[:, :TARGET_LEN]
            print(f" Cropped: {f} ({duration:.2f}s → 3s)")

        out_path = os.path.join(OUTPUT_DIR, f)
        torchaudio.save(out_path, signal, SAMPLE_RATE)
        processed += 1

print(f"\n Done. Processed: {processed}, Skipped: {skipped}")


 Cropped: CDBK21190472333548.wav (3.60s → 3s)
 Padded: XKQP35918600326135.wav (2.21s → 3s)
 Cropped: VMDF13298681926195.wav (3.25s → 3s)
 Padded: HBNG93880431156761.wav (2.12s → 3s)
 Padded: MRPQ70083135550803.wav (2.92s → 3s)
 Cropped: FQIQ55388965219586.wav (3.45s → 3s)
 Cropped: HAXS98131851215726.wav (3.73s → 3s)
 Padded: SNGI37561323315110.wav (2.22s → 3s)
 Cropped: RMPZ52385837079958.wav (5.11s → 3s)
 Cropped: OEUO90021824730237.wav (3.67s → 3s)
 Padded: BHOI53545903254710.wav (2.71s → 3s)
 Cropped: IPVA96902081081101.wav (4.33s → 3s)
 Cropped: KEWB81814152183365.wav (3.73s → 3s)
 Cropped: CJAZ76020257148331.wav (3.29s → 3s)
 Cropped: TEXI73706670090206.wav (3.05s → 3s)
 Padded: ODVG19388574978042.wav (1.82s → 3s)
 Cropped: OVGB49973977166608.wav (3.91s → 3s)
 Cropped: APQC41609456884992.wav (3.05s → 3s)
 Padded: ANKS73776413254295.wav (2.26s → 3s)
 Cropped: VOVB04344857970405.wav (3.12s → 3s)
 Padded: KJXM56485699127338.wav (1.81s → 3s)
 Cropped: TUOD23349254803230.wav (3.08s → 

In [5]:
import IPython.display as ipd

# try random file to ensure from preprocessing

audio, sr = librosa.load("/kaggle/working/preprocessed_train/aew/arctic_a0003.wav", sr=16000)
print(audio.shape, sr)
ipd.Audio(audio, rate=sr)

(48000,) 16000


In [6]:
import pandas as pd

test_meta = pd.read_csv("/kaggle/input/speaker-recognition-cmu-arctic/test_full.csv")
train_meta  = pd.read_csv("/kaggle/input/speaker-recognition-cmu-arctic/train.csv")

print("Train.csv:")
print(train_meta.head())

print("\nTest_full.csv:")
print(test_meta.head())

print("\nTrain columns:", train_meta.columns.tolist())
print("Test columns:", test_meta.columns.tolist())

Train.csv:
          id                   file_path  \
0  rxr_a0591  train/rxr/arctic_a0591.wav   
1  rxr_a0403  train/rxr/arctic_a0403.wav   
2  ljm_a0059  train/ljm/arctic_a0059.wav   
3  jmk_a0134  train/jmk/arctic_a0134.wav   
4  rms_b0067  train/rms/arctic_b0067.wav   

                                              speech speaker  
0                     We are both children together.     rxr  
1    His newborn cunning gave him poise and control.     rxr  
2                His immaculate appearance was gone.     ljm  
3                He obeyed the pressure of her hand.     jmk  
4  Below him the shadow was broken into a pool of...     rms  

Test_full.csv:
                   id                    file_path  \
0  OELV49874079341496  test/OELV49874079341496.wav   
1  ELDT67178766030957  test/ELDT67178766030957.wav   
2  GWMS82652863581753  test/GWMS82652863581753.wav   
3  UAPD90278002373588  test/UAPD90278002373588.wav   
4  IHIY50549582963537  test/IHIY50549582963537.wav   

     

# FEATURE EXTRACTION (MFCC)

In [7]:
def extract_features_from_csv(csv_file, base_dir, sr=16000, duration=3, n_mfcc=40):
    
    df = pd.read_csv(csv_file)
    features = []
    labels = []
    
    for _, row in df.iterrows():
        file_path = os.path.join(base_dir, row["file_path"])
        speaker   = row["speaker"]

        try:
            y, _ = librosa.load(file_path, sr=sr, duration=duration)
            
            if len(y) < sr * duration:
                pad_len = sr * duration - len(y)
                y = np.pad(y, (0, pad_len), mode="constant")
            
            # MFCC
            mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=n_mfcc)
            mfcc = np.mean(mfcc.T, axis=0)

            features.append(mfcc)
            labels.append(speaker)
        
        except Exception as e:
            print(f"Error loading {file_path}: {e}")
    
    return np.array(features), np.array(labels)

X_train, y_train = extract_features_from_csv(
    "/kaggle/input/speaker-recognition-cmu-arctic/train.csv",
    base_dir="/kaggle/input/speaker-recognition-cmu-arctic"
)

X_test, y_test = extract_features_from_csv(
    "/kaggle/input/speaker-recognition-cmu-arctic/test_full.csv",
    base_dir="/kaggle/input/speaker-recognition-cmu-arctic"
)

# SPLIT DATA AND PERPROCESSING

In [8]:
X_train, y_train = extract_features_from_csv("/kaggle/input/speaker-recognition-cmu-arctic/train.csv", base_dir="/kaggle/input/speaker-recognition-cmu-arctic")
X_test, y_test   = extract_features_from_csv("/kaggle/input/speaker-recognition-cmu-arctic/test_full.csv", base_dir="/kaggle/input/speaker-recognition-cmu-arctic")

In [9]:
encoder = LabelEncoder()

y_train = encoder.fit_transform(y_train)
y_test  = encoder.transform(y_test)

In [10]:
print("X_train shape:", X_train.shape)
print("y_train shape:", y_train.shape)
print("X_test shape:", X_test.shape)
print("y_test shape:", y_test.shape)

X_train shape: (12466, 40)
y_train shape: (12466,)
X_test shape: (3117, 40)
y_test shape: (3117,)


In [12]:
X_train = X_train.reshape(X_train.shape[0], X_train.shape[1], 1)
X_test = X_test.reshape(X_test.shape[0], X_test.shape[1], 1)


In [13]:
print("\nFirst 5 labels (y_train):", y_train[:5])
print("First 5 labels (y_test):", y_test[:5])


First 5 labels (y_train): [15 15 12 10 14]
First 5 labels (y_test): [ 0  7  3 10  6]


In [14]:
print("\nUnique speakers in train:", len(set(y_train)))
print("Unique speakers in test:", len(set(y_test)))


Unique speakers in train: 18
Unique speakers in test: 18


# MODEL SET UP

## primary layers Conv1D

In [28]:
from tensorflow.keras import layers, models

def create_embedding_model_with_cnn(input_shape, embedding_dim=256):
    model = models.Sequential([
        layers.Input(shape=input_shape),
        
        layers.Conv1D(filters=64, kernel_size=3, activation='relu', padding='same'),
        layers.MaxPooling1D(pool_size=2),
        layers.Dropout(0.2),
        
        layers.Conv1D(filters=128, kernel_size=3, activation='relu', padding='same'),
        layers.MaxPooling1D(pool_size=2),
        layers.Dropout(0.2),
        
        layers.Conv1D(filters=256, kernel_size=3, activation='relu', padding='same'),
        layers.MaxPooling1D(pool_size=2),
        layers.Dropout(0.2),
        
        
        layers.Flatten(),
        layers.Dense(256, activation="relu"),
        layers.Dropout(0.3),
        layers.Dense(128, activation="relu"),
        layers.Dropout(0.3),
        layers.Dense(embedding_dim)
    ])
    return model


## set triplet loss

In [29]:
import tensorflow.keras.backend as K

def triplet_loss(alpha=0.2):
    def loss(y_true, y_pred):
        total_leng = y_pred.shape.as_list()[-1] // 3

        anchor   = y_pred[:, 0:total_leng]
        positive = y_pred[:, total_leng:2*total_leng]
        negative = y_pred[:, 2*total_leng:]

        pos_dist = K.sum(K.square(anchor - positive), axis=1)
        neg_dist = K.sum(K.square(anchor - negative), axis=1)

        basic_loss = pos_dist - neg_dist + alpha
        loss = K.maximum(basic_loss, 0.0)
        return loss
    return loss

## embedding

In [65]:
import tensorflow as tf

def triplet_accuracy_metric(y_true, y_pred):
  
    # (batch_size, embedding_dim*3)
    embedding_dim = tf.shape(y_pred)[1] // 3
    anchor = y_pred[:, :embedding_dim]
    positive = y_pred[:, embedding_dim:2*embedding_dim]
    negative = y_pred[:, 2*embedding_dim:]

    # calculate distances
    d_pos = tf.norm(anchor - positive, axis=1)
    d_neg = tf.norm(anchor - negative, axis=1)

    # check triplet
    correct = tf.cast(d_pos < d_neg, tf.float32)
    return tf.reduce_mean(correct)


In [63]:
def build_siamese_model(input_shape, embedding_dim=256, alpha=0.2):
    embedding_model = create_embedding_model_with_cnn(input_shape, embedding_dim)

    input_anchor   = layers.Input(shape=input_shape, name="anchor")
    input_positive = layers.Input(shape=input_shape, name="positive")
    input_negative = layers.Input(shape=input_shape, name="negative")

    emb_anchor   = embedding_model(input_anchor)
    emb_positive = embedding_model(input_positive)
    emb_negative = embedding_model(input_negative)

    merged_output = layers.concatenate([emb_anchor, emb_positive, emb_negative], axis=1)

    model = models.Model(inputs=[input_anchor, input_positive, input_negative], outputs=merged_output)
    
    model.compile(
        optimizer=Adam(learning_rate=0.0001),
        loss=triplet_loss(alpha),
        metrics=[triplet_accuracy_metric]
    )
    return model, embedding_model

## **Summary**

build_siamese_model constructs a model that takes three audio recordings as input: an Anchor (A), a Positive (P), and a Negative (N).

Each input recording is passed through the same shared network to produce an embedding (a fixed-size vector representation).

We combine the three embeddings and compute the Triplet Loss on them during training.

After training, we extract and use the embedding_model (the shared network alone) as a tool to convert any audio recording into its embedding vector.

To perform speaker verification or identification, we compare embeddings (e.g., using cosine similarity or Euclidean distance) to decide whether two recordings belong to the same speaker.

In [35]:
import random

def generate_triplets(features, labels, num_triplets=10000):
    
    triplets = []
    unique_labels = np.unique(labels)

    for _ in range(num_triplets):
        pos_label = random.choice(unique_labels)

        pos_indices = np.where(labels == pos_label)[0]
        anchor_idx, positive_idx = np.random.choice(pos_indices, 2, replace=True)

        neg_label = random.choice(unique_labels[unique_labels != pos_label])
        neg_indices = np.where(labels == neg_label)[0]
        negative_idx = np.random.choice(neg_indices)

        triplets.append([features[anchor_idx], features[positive_idx], features[negative_idx]])

    A, P, N = zip(*triplets)
    return np.array(A), np.array(P), np.array(N)

In [54]:
A_train, P_train, N_train = generate_triplets(X_train, y_train, num_triplets=1000)

In [ ]:
earlyStop = EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True)
reduceLR = ReduceLROnPlateau(monitor='val_loss', patience=7, factor=0.1)
modelPoints = ModelCheckpoint('/kaggle/working/model.h5', save_best_only=True)

## Train/Evaluate/triplet loss accuracy

In [ ]:
input_shape = A_train.shape[1:]
siamese_model, embedding_model = build_siamese_model(input_shape)

history = siamese_model.fit(
    [A_train, P_train, N_train],
    np.zeros(len(A_train)),
    batch_size=64,
    validation_split=0.2,
    epochs=15,
    callbacks=[earlyStop, reduceLR, modelPoints]
)

Epoch 1/15
13/13 ━━━━━━━━━━━━━━━━━━━━ 9s 128ms/step - loss: 572.6066 - triplet_accuracy_metric: 0.5127 - val_loss: 0.0000e+00 - val_triplet_accuracy_metric: 1.0000
Epoch 2/15
13/13 ━━━━━━━━━━━━━━━━━━━━ 1s 57ms/step - loss: 282.4101 - triplet_accuracy_metric: 0.5002 - val_loss: 0.0000e+00 - val_triplet_accuracy_metric: 1.0000
Epoch 3/15
13/13 ━━━━━━━━━━━━━━━━━━━━ 1s 57ms/step - loss: 153.1959 - triplet_accuracy_metric: 0.5245 - val_loss: 0.0000e+00 - val_triplet_accuracy_metric: 1.0000
Epoch 4/15
13/13 ━━━━━━━━━━━━━━━━━━━━ 1s 65ms/step - loss: 109.2871 - triplet_accuracy_metric: 0.4924 - val_loss: 0.0000e+00 - val_triplet_accuracy_metric: 1.0000
Epoch 5/15
13/13 ━━━━━━━━━━━━━━━━━━━━ 1s 62ms/step - loss: 91.7863 - triplet_accuracy_metric: 0.4422 - val_loss: 0.0000e+00 - val_triplet_accuracy_metric: 1.0000
Epoch 6/15
13/13 ━━━━━━━━━━━━━━━━━━━━ 1s 60ms/step - loss: 70.3714 - triplet_accuracy_metric: 0.4837 - val_loss: 0.0000e+00 - val_triplet_accuracy_metric: 1.0000
Epoch 7/15
13/13 ━━━━━━